## 🛠️ Section 0: Kaggle Workspace Setup & Code Cloning


In [ ]:
# 0. Clone mã nguồn dự án vào thư mục /kaggle/working/r2AI_2026
import os
import sys
import shutil
import subprocess
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "r2AI_2026"
REPO_URL = "https://github.com/duymcminh/r2AI_2026.git"

if Path("/kaggle/working").exists():
    print("🚀 Phát hiện môi trường Kaggle! Đang chuẩn bị thư mục làm việc...")
    if not REPO_DIR.exists():
        print(f"📥 Đang clone repository từ {REPO_URL} vào {REPO_DIR}...")
        res = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"⚠️ Lỗi Git Clone: {res.stderr.strip()}")
            dataset_candidates = list(Path("/kaggle/input").glob("**/r2AI_2026")) if Path("/kaggle/input").exists() else []
            if dataset_candidates:
                src_path = dataset_candidates[0]
                print(f"📦 Tìm thấy mã nguồn trong Kaggle Input Dataset: {src_path}. Đang sao chép sang {REPO_DIR}...")
                shutil.copytree(src_path, REPO_DIR, dirs_exist_ok=True)
    if REPO_DIR.exists():
        os.chdir(str(REPO_DIR))
        if str(REPO_DIR) not in sys.path:
            sys.path.insert(0, str(REPO_DIR))
        print(f"✅ Đã chuyển thư mục làm việc: {os.getcwd()}")
    else:
        print(f"⚠️ Không tìm thấy thư mục {REPO_DIR}. Tiếp tục với thư mục làm việc mặc định: {os.getcwd()}")
else:
    local_root = Path("..").resolve()
    if str(local_root) not in sys.path:
        sys.path.insert(0, str(local_root))
    print(f"💻 Đang chạy trên môi trường Local: {os.getcwd()}")


## 📥 Section 0.1: Installing Dependencies on Kaggle


In [ ]:
# 1. Cài đặt các gói phụ thuộc dự án (Bao gồm Qdrant Client, Sentence Transformers, BM25, LangGraph, ...)
print("📥 Đang cài đặt Python dependencies cho RAG Search Engine...")
!pip install -q \
    qdrant-client \
    sentence-transformers \
    rank-bm25 \
    thefuzz \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    langchain-openai>=0.2.0 \
    json-repair>=0.30.0 \
    openpyxl>=3.1.0 \
    tabulate>=0.9.0 \
    pyyaml>=6.0

print("✅ Cài đặt dependencies thành công!")


## 🔗 Section 0.2: Linking Qdrant Local DB & Datasets from Kaggle Path


In [ ]:
# 2. Dò tìm và liên kết trực tiếp Qdrant Local DB từ đường dẫn chỉ định trên Kaggle
import os
from pathlib import Path

KAGGLE_EXPLICIT_DIR = Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data")

if Path("/kaggle/input").exists():
    print("🔍 Đang kiểm tra Kaggle Input Dataset cho Qdrant DB...")
    dataset_dir = None
    if (KAGGLE_EXPLICIT_DIR / "qdrant_local_db").exists():
        dataset_dir = KAGGLE_EXPLICIT_DIR
        print(f"📦 Đã tìm thấy Qdrant DB tại đường dẫn chỉ định: {KAGGLE_EXPLICIT_DIR / 'qdrant_local_db'}")
    else:
        qdrant_found = list(Path("/kaggle/input").glob("**/qdrant_local_db"))
        if qdrant_found:
            dataset_dir = qdrant_found[0].parent
            print(f"📦 Đã tìm thấy Qdrant DB bằng wildcard tại: {qdrant_found[0]}")
            
    if dataset_dir:
        rag_module_dir = Path("rag_module").resolve()
        rag_module_dir.mkdir(exist_ok=True)
        
        # Tạo symlink tới qdrant_local_db, bm25_index.pkl, code_stock.csv, ViFinQA
        for item in ["qdrant_local_db", "bm25_index.pkl", "code_stock.csv", "ViFinQA"]:
            src = dataset_dir / item
            dst = rag_module_dir / item
            if src.exists() and not dst.exists():
                try:
                    os.symlink(src, dst)
                    print(f"   🔗 Created symlink: {dst} -> {src}")
                except Exception as e:
                    print(f"   ⚠️ Cannot symlink {item}: {e}")
else:
    print("💻 Chạy local, sử dụng dữ liệu có sẵn tại rag_module/")


## ⚙️ Section 1: Search Engine Initialization & Resource Load


In [ ]:
# 3. Nạp Search Engine & Tài nguyên (Qdrant + SentenceTransformers + BM25)
import rag_module.search_engine as se
se._ensure_resources()
print("✅ Search Engine resources loaded successfully!")


## 📋 Section 2: Sample Questions List


In [ ]:
# Danh sách 20 câu hỏi kiểm thử mẫu
SAMPLE_QUESTIONS = [
    (1, "Lãi tiền gửi năm 2018 của công ty mẹ CTCP Hàng không Vietjet (VJC) là bao nhiêu triệu đồng?"),
    (2, "Số dư cho vay khách hàng ngành Thương mại của công ty mẹ Ngân hàng TMCP Á Châu (ACB) cuối năm 2022 là bao nhiêu triệu đồng?"),
    (3, "Chi phí dự phòng của Ngân hàng TMCP Sài Gòn Tài Lộc trong năm 2020 là bao nhiêu triệu đồng?"),
    (4, "Lợi nhuận sau thuế của CTCP Chứng khoán FPT năm 2023 là bao nhiêu tỷ đồng?"),
    (5, "Chi phí phạt của công ty mẹ SCR năm 2017 là bao nhiêu tỷ đồng?"),
    (6, "Lưu chuyển tiền thuần từ hoạt động kinh doanh của công ty mẹ VSC trong năm 2017 là bao nhiêu tỷ đồng?"),
    (7, "Quỹ khen thưởng, phúc lợi của HT1 cuối năm 2019 là bao nhiêu tỷ đồng?"),
    (8, "Chi phí lương và các khoản khác theo lương của công ty mẹ CTCP Chứng khoán FPT trong năm 2021 là bao nhiêu tỷ đồng?"),
    (9, "Chi phí khác của SAM năm 2023 là bao nhiêu triệu đồng?"),
    (10, "Chi phí tài chính của công ty mẹ CTCP Phát triển Sunshine Homes năm 2021 là bao nhiêu triệu đồng?"),
    (11, "Số dư tiền gửi tại các TCTD khác cuối năm 2016 của Ngân hàng TMCP Đầu tư và Phát triển Việt Nam (BID) là bao nhiêu triệu đồng?"),
    (12, "Vốn cổ phần đã phát hành của công ty mẹ VGT là bao nhiêu nghìn tỷ đồng vào cuối năm 2024?"),
    (13, "Tiền và các khoản tương đương tiền của công ty mẹ Tổng Công ty cổ phần Bia - Rượu - Nước giải khát Sài Gòn (SAB) vào cuối năm 2016 là bao nhiêu tỷ đồng?"),
    (14, "Chi phí quản lý doanh nghiệp năm 2025 của công ty mẹ ASM là bao nhiêu triệu đồng?"),
    (15, "Thù lao của thành viên HĐQT Chu Thị Bình tại công ty mẹ MPC năm 2021 là bao nhiêu triệu đồng?"),
    (16, "Số dư vay ngắn hạn của công ty mẹ CEO cuối năm 2025 là bao nhiêu tỷ đồng?"),
    (17, "Lãi thuần từ hoạt động dịch vụ của Ngân hàng TMCP Sài Gòn - Hà Nội (SHB) năm 2018 là bao nhiêu triệu đồng?"),
    (18, "Vốn chủ sở hữu của FIT là bao nhiêu tỷ đồng vào ngày 31/12/2015?"),
    (19, "Tổng tỷ lệ quyền biểu quyết của công ty mẹ CTCP Đầu tư Hạ tầng Giao thông Đèo Cả năm 2023 là bao nhiêu phần trăm?"),
    (20, "Tỷ lệ biểu quyết của Xí nghiệp Liên doanh Visorutex của công ty mẹ GVR đến ngày 31/12/2019 là bao nhiêu %?"),
]

print(f"📋 Đã tải {len(SAMPLE_QUESTIONS)} câu hỏi kiểm thử.")


## 🧠 Section 3: Core Debugger Function & Enhanced Diagnostic Logging


In [ ]:
# Hàm Debug Truy Vấn & Đánh giá Chi Tiết RRF (Chế độ Diagnostic Logging Nâng Cao)
import re
from pathlib import Path

def clean_query_content_from_query(q_text: str, ticker: str, year: str) -> str:
    """
    Làm sạch câu hỏi: Loại bỏ tên công ty, mã chứng khoán, năm, và các từ để hỏi
    để thu được chuỗi nội dung chỉ tiêu cốt lõi (VD: 'Lãi tiền gửi', 'Quỹ khen thưởng, phúc lợi').
    """
    text = q_text
    text = re.sub(r"\([A-Z]{2,5}\)", "", text)
    text = re.sub(r"\b20\d{2}\b", "", text)
    patterns = [
        r"là bao nhiêu.*", r"bao nhiêu.*", r"của công ty mẹ.*", r"của ngân hàng.*",
        r"của ctcp.*", r"của tập đoàn.*", r"của công ty.*", r"vào ngày.*",
        r"đến ngày.*", r"tại ngày.*", r"cuối năm.*", r"đầu năm.*",
        r"trong năm.*", r"năm.*", r"báo cáo tài chính.*", r"báo cáo riêng.*", r"báo cáo hợp nhất.*",
    ]
    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)
        prefix_patterns = [
        r"^\s*tổng\s+số\s+", r"^\s*tổng\s+", r"^\s*số\s+dư\s+",
        r"^\s*giá\s+trị\s+", r"^\s*chỉ\s+tiêu\s+",
    ]
    for pp in prefix_patterns:
        text = re.sub(pp, "", text, flags=re.IGNORECASE)
    cleaned = text.strip(" ,.?:;\t\n")
    return cleaned if len(cleaned) >= 2 else q_text

def debug_query(q_id: int, q_text: str, output_dir: Path) -> str:
    """
    Thực thi tìm kiếm cho 1 câu hỏi với LOGGING CHẨN ĐOÁN CHI TIẾT:
    - Ghi nhận thông tin trích xuất thực thể, chuỗi noi_dung thô vs noi_dung_clean.
    - Thống kê chi tiết thứ hạng Dense Vector, Sparse BM25, và trạng thái nhận 0.1 Substring Bonus.
    - Đánh giá độ khả thi và phân tích nguyên nhân tiềm năng nếu điểm RRF thấp (< 0.05).
    """
    ticker, year, default_rpt = se.parse_query(q_text, se._company_map)
    if not ticker:
        m = re.search(r"\b([A-Z]{3,5})\b", q_text)
        if m:
            ticker = se._resolve_ticker(m.group(1))

    noi_dung_clean = clean_query_content_from_query(q_text, ticker, year)

    # Tìm kiếm trên cả 2 loại báo cáo (report_type=None)
    results = se.search_by_company_and_content(
        company_name=ticker,
        content=noi_dung_clean,
        year=year if year else None,
        report_type=None,
        top_k=100
    )

    log_lines = []
    log_lines.append(f"================================================================================")
    log_lines.append(f"CÂU HỎI ID #{q_id}: {q_text}")
    log_lines.append(f"================================================================================")
    log_lines.append(f"📍 [1. THỰC THỂ TRÍCH XUẤT]: Ticker='{ticker}', Năm='{year}', DefaultReportType='{default_rpt}'")
    log_lines.append(f"🎯 [2. NỘI DUNG TÌM KIẾM]: CleanContent='{noi_dung_clean}' | RawQuery='{q_text}'")
    log_lines.append(f"📊 [3. THỐNG KÊ KẾT QUẢ]: Đã đánh giá {len(results)} bảng ứng viên từ Search Engine")
    
    top1_rrf = results[0].get("rrf_score", 0.0) if results else 0.0
    exact_bonus_triggered = top1_rrf > 0.1
    log_lines.append(f"⚡ [4. CHẨN ĐOÁN KÍCH HOẠT SUBSTRING BONUS]: {'ĐÃ KÍCH HOẠT (RRF > 0.1)' if exact_bonus_triggered else 'CHƯA KÍCH HOẠT (RRF < 0.05 - Cần kiểm tra chuỗi khớp)'}")
    
    log_lines.append(f"-" * 85)
    log_lines.append(f"{'RANK':<5} | {'RRF SCORE':<10} | {'BÁO CÁO':<12} | {'HÀNG NỘI DUNG (CỘT 0) MANG RRF CAO NHẤT':<45} | {'TÊN FILE CSV'}")
    log_lines.append(f"-" * 85)

    for rank, item in enumerate(results, 1):
        rrf = item.get("rrf_score", 0.0)
        rpt = item.get("Loai_Bao_Cao", "N/A")
        sample = str(item.get("matched_sample", "N/A")).replace("\n", " ")
        if len(sample) > 45:
            sample = sample[:42] + "..."
        csv_name = Path(item.get("csv_path", "")).name
        
        bonus_str = " [+0.1 BONUS]" if (noi_dung_clean.lower() in sample.lower()) else ""
        log_lines.append(f"#{rank:<4} | {rrf:<10.6f} | {rpt:<12} | {sample:<45} | {csv_name}")
        log_lines.append(f"       Tên Bảng : {item.get('Ten_Bang', 'N/A')}")
        log_lines.append(f"       Thứ Hạng : Vector/Dense Rank = {item.get('dense_rank', '-')}, BM25/Sparse Rank = {item.get('sparse_rank', '-')}{bonus_str}")
        log_lines.append(f"       Cột Khớp : {item.get('matched_col_name', 'N/A')}")
        log_lines.append("")

    log_text = "\n".join(log_lines)
    
    output_dir.mkdir(parents=True, exist_ok=True)
    txt_file = output_dir / f"query_{q_id}_debug.txt"
    with open(txt_file, "w", encoding="utf-8") as f:
        f.write(log_text)
        
    return log_text

print("✅ Hàm debug_query đã sẵn sàng với Diagnostic Logging Nâng Cao!")



## 📄 Section 4: Master Execution & Exporting Debug Summary


In [ ]:
# Chạy Debug trên 20 Câu hỏi Mẫu & Xuất Báo cáo .txt Tổng hợp
repo_dir = Path("/kaggle/working/r2AI_2026") if Path("/kaggle/working/r2AI_2026").exists() else Path("..").resolve()
output_dir = repo_dir / "debug_outputs"
all_summaries = []

print(f"🚀 Bắt đầu thực thi debug cho {len(SAMPLE_QUESTIONS)} câu hỏi mẫu...")
print(f"📁 Thư mục lưu file .txt log: {output_dir.resolve()}")
print("=" * 60)

for q_id, q_text in SAMPLE_QUESTIONS:
    print(f"⏳ Đang xử lý câu hỏi #{q_id}...")
    summary_text = debug_query(q_id, q_text, output_dir)
    all_summaries.append(summary_text)
    print(f"   ✅ Đã hoàn thành & lưu: query_{q_id}_debug.txt")

# Xuất file tổng hợp 20 câu hỏi
master_summary_file = output_dir / "all_20_queries_debug_summary.txt"
with open(master_summary_file, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_summaries))

print("=" * 60)
print(f"🎉 HOÀN THÀNH TOÀN BỘ 20 CÂU HỎI KIỂM THỬ!")
print(f"📄 File log tổng hợp đã được lưu tại: {master_summary_file.resolve()}")
